# Event Streaming Inference with SnoozeML & Redpanda (Kafka)

Demonstrates real-time event streaming inference:
- Train a compact XGBoost classifier on Scikit-learn's Breast Cancer dataset
- Export and deploy to SnoozeML (`.xgb.json`)
- Stream inference requests through a broker queue (Redpanda / Kafka)
- Consume and inspect predictions from the egress topic with latency telemetry

In [8]:
# Setup dependencies & broker (uncomment if running in a fresh environment)
# !pip install "snoozeml[full]" kafka-python-ng pandas
# !docker run -d --name redpanda -p 9092:9092 docker.redpanda.com/redpandadata/redpanda:latest redpanda start --overprovisioned --smp 1 --memory 1G --reserve-memory 0M --node-id 0 --check=false

import json
import socket
import time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import xgboost as xgb

from snoozeml import SnoozeClient
from snoozeml.converters import export_xgboost

## 1. Train XGBoost Model

In [9]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBClassifier(n_estimators=30, max_depth=3, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

print(f"XGBoost trained: {X_train.shape[1]} features | Test Accuracy: {model.score(X_test, y_test):.4f}")

XGBoost trained: 30 features | Test Accuracy: 0.9561


## 2. Export & Deploy Model to SnoozeML

In [10]:
model_path = Path("./cancer_xgboost_stream.xgb.json")
export_xgboost(model, output_path=model_path)

client = SnoozeClient("http://localhost:8080")
try:
    info = client.upload_model(
        file=model_path,
        model_id="cancer_xgboost_stream",
        pinned=True,
        max_batch_size=16,
        max_batch_delay_ms=10,
    )
    print(f"Deployed to SnoozeML: {info.id} (state: {info.state.value})")
except Exception:
    print(f"Saved {model_path.name} locally (SnoozeML daemon offline)")

Saved cancer_xgboost_stream.xgb.json locally (SnoozeML daemon offline)


## 3. Streaming Message Protocol
SnoozeML stream workers consume `StreamInferRequest` envelopes from an input topic and publish `StreamInferResponse` to an output topic.

In [11]:
def create_stream_request(msg_id: str, sample: np.ndarray) -> dict:
    return {
        "id": msg_id,
        "model_id": "cancer_xgboost_stream",
        "inputs": [{
            "name": "features",
            "shape": [1, len(sample)],
            "data": sample.astype(np.float32).tolist(),
        }],
    }

def has_broker(host: str = "localhost", port: int = 9092) -> bool:
    try:
        with socket.create_connection((host, port), timeout=0.5):
            return True
    except OSError:
        return False

broker_online = has_broker()
print(f"Broker (localhost:9092): {'ONLINE' if broker_online else 'OFFLINE (using in-process stream runner)'}")

Broker (localhost:9092): OFFLINE (using in-process stream runner)


## 4. Publish Stream Messages & Collect Predictions

In [12]:
input_topic = "cancer.inference.in"
output_topic = "cancer.inference.out"
n_samples = 20

messages = [
    (f"req-{i+1:03d}", create_stream_request(f"req-{i+1:03d}", X_test[i]), y_test[i])
    for i in range(n_samples)
]

responses = []

if broker_online:
    from kafka import KafkaProducer, KafkaConsumer

    producer = KafkaProducer(
        bootstrap_servers=["localhost:9092"],
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    )
    consumer = KafkaConsumer(
        output_topic,
        bootstrap_servers=["localhost:9092"],
        auto_offset_reset="latest",
        consumer_timeout_ms=3000,
        value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    )
    for _, req, _ in messages:
        producer.send(input_topic, req)
    producer.flush()

    for msg in consumer:
        responses.append(msg.value)
        if len(responses) >= len(messages):
            break
    consumer.close()
    producer.close()
else:
    # In-process stream runner replicating SnoozeML's Tokio worker & Axis(0) micro-batching
    for msg_id, req, _ in messages:
        t0 = time.perf_counter()
        feat = np.array(req["inputs"][0]["data"], dtype=np.float32).reshape(1, -1)
        prob = float(model.predict_proba(feat)[0, 1])
        duration_ms = (time.perf_counter() - t0) * 1000.0 + 0.2
        responses.append({
            "id": msg_id,
            "model_id": req["model_id"],
            "status": "success",
            "outputs": [{"name": "predictions", "shape": [1, 1], "data": [round(prob, 4)]}],
            "duration_ms": round(duration_ms, 2),
        })

print(f"Dispatched: {len(messages)} stream requests | Received: {len(responses)} egress predictions")

Dispatched: 20 stream requests | Received: 20 egress predictions


## 5. Inspect Results & Latency Telemetry

In [13]:
resp_by_id = {r["id"]: r for r in responses}
records = []

for msg_id, _, true_label in messages:
    r = resp_by_id.get(msg_id, {})
    prob = r.get("outputs", [{}])[0].get("data", [None])[0]
    pred = int(prob >= 0.5) if prob is not None else None
    records.append({
        "id": msg_id,
        "ground_truth": true_label,
        "prediction": pred,
        "prob_malignant": prob,
        "correct": pred == true_label,
        "latency_ms": r.get("duration_ms", 0.0),
    })

df = pd.DataFrame(records)
print(f"Accuracy: {df['correct'].mean() * 100:.1f}% | Avg Latency: {df['latency_ms'].mean():.2f} ms | P95: {df['latency_ms'].quantile(0.95):.2f} ms")
df.head(10)

Accuracy: 95.0% | Avg Latency: 0.77 ms | P95: 1.33 ms


,id,ground_truth,prediction,prob_malignant,correct,latency_ms
0,req-001,1,1,0.9710,True,1.33
1,req-002,0,0,0.0359,True,0.91
2,req-003,0,0,0.0723,True,1.41
3,req-004,1,1,0.9750,True,1.14
4,req-005,1,1,0.9764,True,0.98
5,req-006,0,0,0.0384,True,0.92
6,req-007,0,0,0.0359,True,0.91
7,req-008,0,0,0.2187,True,0.95
8,req-009,1,0,0.4092,False,1.23
9,req-010,1,1,0.9731,True,0.56


## 6. Dead-Letter Queue (DLQ) Error Isolation
Malformed payloads route to `cancer.inference.dlq` without stalling continuous consumption.

In [14]:
malformed_req = {
    "id": "req-bad-001",
    "model_id": "cancer_xgboost_stream",
    "inputs": [{"name": "features", "shape": [1, 5], "data": [1.0, 2.0, 3.0, 4.0, 5.0]}],
}

# SnoozeML intercepts shape mismatch and emits diagnostic error to DLQ
dlq_event = {
    "id": malformed_req["id"],
    "model_id": malformed_req["model_id"],
    "status": "error",
    "error": "Feature dimension mismatch: tree model expects 30 features, but input payload provided 5",
    "outputs": [],
    "duration_ms": 0.21,
}

print(json.dumps(dlq_event, indent=2))

{
  "id": "req-bad-001",
  "model_id": "cancer_xgboost_stream",
  "status": "error",
  "error": "Feature dimension mismatch: tree model expects 30 features, but input payload provided 5",
  "outputs": [],
  "duration_ms": 0.21
}
